# Critical-reasoning SLM — SFT of **Qwen3.5-0.8B** (Colab)

Fine-tunes **Qwen3.5-0.8B** (QLoRA) to reliably do one thing: given a self-contained
logic problem, produce a short premise-grounded rationale and commit to exactly one
credited answer in the required format — an option letter (MCQ) or a fixed
entailment label **True / False / Uncertain|Unknown**, including the neutral
"does-not-follow" class (the hard part).

**Data is pre-built** (`data/sft_train.jsonl`, ~8k rows): every completion is a
**house-style verbalized trace** produced by one verbalizer at build time
(`build_corpus.py`) — ProverQA uses its prover-derived chain restyled; FOLIO/LogicNLI
use grounded entailment/neutral reasoning; the MCQ families cite the credited option.
No teacher call at train time. Two regimes, never merged:

- **entailment core** — FOLIO, LogicNLI, ProverQA → FRQ label `True/False/neutral`
- **argument satellite** — ReClor (flaw/assumption/weaken), adversarial ARCT, LogiQA → MCQ

Held-out eval is a hashed 20% slice of **every** family (`data/eval_items.json`).
See `../PLAN.md` → "Day 3+" for the full spec and the ~10k budget.

**How to run:** upload this whole `colab/` folder to the Colab working directory
(or mount Drive and `cd` into it), set the runtime to a **GPU** (Runtime → Change
runtime type → T4), then run the cells top to bottom.

In [ ]:
#@title 1. Install dependencies
# Colab already ships a CUDA torch build. Add the SFT stack + 4-bit QLoRA.
!pip -q install "transformers>=4.51" "peft>=0.11" "accelerate>=0.30" "bitsandbytes>=0.43" "safetensors>=0.4"
# Qwen3.5 uses LINEAR-ATTENTION layers; its fast path needs these. Optional — if the
# wheels don't build on your runtime that's fine, the model falls back to a slower,
# more memory-hungry torch path (then lower BATCH_SIZE / MAX_LEN in the Config cell).
!pip -q install flash-linear-attention causal-conv1d 2>/dev/null || echo "fla/causal-conv1d not installed — using torch fallback (reduce BATCH_SIZE/MAX_LEN if OOM)"

import torch, os
print("torch", torch.__version__, "| CUDA available:", torch.cuda.is_available(),
      "|", (torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only"))
assert torch.cuda.is_available(), "Set Runtime -> Change runtime type -> GPU (T4)."

In [ ]:
#@title 2. Locate the folder (must contain slm_core.py + data/sft_train.jsonl)
# If you uploaded the folder to /content, or mounted Drive, point PROJECT_DIR at it.
import os, sys
from pathlib import Path

PROJECT_DIR = ""  #@param {type:"string"}  # leave blank to auto-detect

def _find():
    if PROJECT_DIR and (Path(PROJECT_DIR) / "slm_core.py").exists():
        return Path(PROJECT_DIR)
    for c in [Path.cwd(), Path("/content"), Path("/content/colab"),
              *Path("/content").glob("**/slm_core.py")]:
        c = c if c.is_dir() else c.parent
        if (c / "slm_core.py").exists() and (c / "data" / "sft_train.jsonl").exists():
            return c
    raise FileNotFoundError(
        "Could not find slm_core.py + data/sft_train.jsonl. Upload the colab/ folder "
        "and set PROJECT_DIR above (e.g. /content/colab).")

ROOT = _find()
os.chdir(ROOT); sys.path.insert(0, str(ROOT))
print("project dir:", ROOT)

import json, slm_core as S
from collections import Counter
train_rows = S.load_sft_rows()
eval_items = S.load_eval_items()
meta = json.loads((ROOT / "data" / "corpus_meta.json").read_text())
print(f"train rows: {len(train_rows)} | eval items: {len(eval_items)}")
print("  train by family:", meta["train_by_family"])
print("  train by format:", meta["train_by_format_type"])
print("  eval  by family:", meta["eval_by_family"], "| neutral-gold:", meta["eval_neutral_gold"])

In [ ]:
#@title 3. Config
MODEL_ID       = "Qwen/Qwen3.5-0.8B"   #@param {type:"string"}
EPOCHS         = 3        #@param {type:"integer"}
BATCH_SIZE     = 8        #@param {type:"integer"}
LR             = 2e-4     #@param {type:"number"}
LORA_R         = 16       #@param {type:"integer"}
LORA_ALPHA     = 32       #@param {type:"integer"}
MAX_LEN        = 1024     #@param {type:"integer"}
USE_4BIT       = True     #@param {type:"boolean"}
OUT_DIR        = "runs/qwen35_0p8b"  #@param {type:"string"}
LORA_TARGETS   = ["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"]

# Data (train rows + held-out eval items) is pre-built in data/ by build_corpus.py,
# so there is no MODE switch here: each row already carries its own regime
# (entailment -> frq label, argument -> mcq letter).
# NOTE: verify the exact HF id at https://huggingface.co/Qwen — if Qwen3.5-0.8B is
# gated/renamed, swap MODEL_ID (e.g. "Qwen/Qwen3-0.6B" is a known-good fallback).
print("model:", MODEL_ID, "| epochs:", EPOCHS, "| batch:", BATCH_SIZE, "| 4bit:", USE_4BIT)

In [ ]:
#@title 4. Inspect the pre-built SFT data (house-style verbalized traces)
# train_rows / eval_items were loaded in cell 2. Rows are {prompt, completion, gold, ...}.
print(f"train rows: {len(train_rows)}  ",
      dict(Counter((r['family'], r['mode']) for r in train_rows)))
print("trace source:", meta["trace_source"], "| LGMT follow-ups:", meta["n_lgmt"])

ex = next(r for r in train_rows if r["family"] == "logicnli" and r["gold"] in ("Unknown", "Uncertain"))
print("\n--- example NEUTRAL row (the hard class) ---")
print("PROMPT:\n", ex["prompt"][:280], "\n\nCOMPLETION:\n", ex["completion"])

In [ ]:
#@title 5. Load Qwen3.5-0.8B (+ 4-bit QLoRA) and attach LoRA
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

tok = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

quant_config = None
if USE_4BIT:
    from transformers import BitsAndBytesConfig
    quant_config = BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.bfloat16)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, trust_remote_code=True, quantization_config=quant_config,
    torch_dtype=torch.bfloat16, device_map="auto")
if quant_config is not None:
    model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
model.config.use_cache = False

model = get_peft_model(model, LoraConfig(
    r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=0.05, bias="none",
    task_type="CAUSAL_LM", target_modules=LORA_TARGETS))
model.print_trainable_parameters()

In [ ]:
#@title 6. Tokenize with completion-only loss masking (train only on the answer)
def format_and_tokenize(prompt, completion):
    def chat(add_gen):
        try:
            return tok.apply_chat_template([{"role": "user", "content": prompt}],
                tokenize=False, add_generation_prompt=add_gen, enable_thinking=False)
        except TypeError:
            return tok.apply_chat_template([{"role": "user", "content": prompt}],
                tokenize=False, add_generation_prompt=add_gen)
    prefix = chat(True)
    full = prefix + completion + (tok.eos_token or "")
    prefix_ids = tok(prefix, add_special_tokens=False)["input_ids"]
    full_ids = tok(full, add_special_tokens=False)["input_ids"][:MAX_LEN]
    labels = list(full_ids)
    for i in range(min(len(prefix_ids), len(labels))):
        labels[i] = -100          # mask the prompt; learn only the completion
    return full_ids, labels

encoded = [format_and_tokenize(r["prompt"], r["completion"]) for r in train_rows]
encoded = [(x, y) for x, y in encoded if any(t != -100 for t in y)]
print(f"encoded {len(encoded)} examples (max_len {MAX_LEN})")

In [ ]:
#@title 7. Train (LoRA SFT loop)
import time
from torch.utils.data import DataLoader

def collate(batch):
    maxlen = max(len(x) for x, _ in batch); pad = tok.pad_token_id
    ii, ll, aa = [], [], []
    for x, y in batch:
        n = maxlen - len(x)
        ii.append(x + [pad] * n); ll.append(y + [-100] * n); aa.append([1]*len(x) + [0]*n)
    return torch.tensor(ii), torch.tensor(ll), torch.tensor(aa)

dl = DataLoader(encoded, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate)
optim = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=LR)
model.train(); t0, step, losses = time.time(), 0, []
for epoch in range(EPOCHS):
    for input_ids, labels, attn in dl:
        input_ids, labels, attn = input_ids.cuda(), labels.cuda(), attn.cuda()
        out = model(input_ids=input_ids, attention_mask=attn, labels=labels)
        out.loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optim.step(); optim.zero_grad(set_to_none=True)
        losses.append(float(out.loss.detach())); step += 1
        if step % 10 == 0 or step == 1:
            print(f"  step {step} epoch {epoch} loss {losses[-1]:.4f} "
                  f"({time.time()-t0:.0f}s)", flush=True)

from pathlib import Path
adapter_dir = Path(OUT_DIR) / "adapter"; adapter_dir.mkdir(parents=True, exist_ok=True)
model.save_pretrained(str(adapter_dir)); tok.save_pretrained(str(adapter_dir))
print(f"\nsaved adapter -> {adapter_dir} | steps {step} | "
      f"loss {losses[0]:.3f} -> {losses[-1]:.3f} | {time.time()-t0:.0f}s")

In [ ]:
#@title 8. Base-vs-tuned eval on the held-out split (deterministic grading)
import torch
from collections import defaultdict

def eval_mode(it):
    # entailment core -> FRQ label; argument satellite -> MCQ letter (its native regime)
    return "frq" if it["family"] in S.ENTAILMENT_FAMILIES else "mcq"

@torch.no_grad()
def generate(m, it, mode, max_new_tokens=512):   # 512 > longest ProverQA chain (~381 tok)
    prompt = S.build_prompt(it, mode)
    try:
        text = tok.apply_chat_template([{"role": "user", "content": prompt}],
            tokenize=False, add_generation_prompt=True, enable_thinking=False)
    except TypeError:
        text = tok.apply_chat_template([{"role": "user", "content": prompt}],
            tokenize=False, add_generation_prompt=True)
    enc = tok(text, return_tensors="pt").to(m.device)
    out = m.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False,
                     pad_token_id=(tok.pad_token_id or tok.eos_token_id))
    return tok.decode(out[0][enc["input_ids"].shape[1]:], skip_special_tokens=True)

def run_condition(m, label):
    b = defaultdict(lambda: {"n":0,"parse":0,"correct":0,"neu_n":0,"neu_c":0})
    for it in eval_items:
        mode = eval_mode(it)
        if mode == "mcq" and not S.has_mc(it):
            continue
        g = S.grade(it, generate(m, it, mode), mode)   # add judge_model=<cheap id> to enable the fallback
        if g["needs_judge"]:            # (no open-ended FRQ in this build -> never hit)
            continue
        k = b[(it["family"], mode)]
        k["n"] += 1; k["parse"] += g["parseable"]; k["correct"] += bool(g["correct"])
        if S.is_neutral_gold(it, mode):
            k["neu_n"] += 1; k["neu_c"] += bool(g["correct"])
    tot = {"n":0,"parse":0,"correct":0,"neu_n":0,"neu_c":0}
    for v in b.values():
        for kk in tot: tot[kk]+=v[kk]
    pct = lambda a,d: round(100*a/d,1) if d else None
    print(f"[{label}] n={tot['n']}  parse={pct(tot['parse'],tot['n'])}%  "
          f"acc={pct(tot['correct'],tot['n'])}%  neutral_recall={pct(tot['neu_c'],tot['neu_n'])}%")
    return b

# eval is training-free: re-enable the KV cache and turn OFF gradient checkpointing
# (both training-only) so generation is fast.
model.eval()
model.config.use_cache = True
try:
    model.gradient_checkpointing_disable()
except Exception:
    pass

# base (LoRA disabled) vs tuned (LoRA enabled) — same weights, isolates the adapter
with model.disable_adapter():
    base_b = run_condition(model, "base ")
tuned_b = run_condition(model, "tuned")

print("\nby family:mode  (base_acc -> tuned_acc, n)   [neutral recall in brackets where applicable]")
pct = lambda a,d: round(100*a/d,1) if d else None
for k in sorted(set(base_b)|set(tuned_b)):
    bb, tt = base_b[k], tuned_b[k]
    neu = f"  neutral {pct(tt['neu_c'],tt['neu_n'])}%" if tt['neu_n'] else ""
    print(f"  {k[0]}:{k[1]:<3}  {pct(bb['correct'],bb['n'])}% -> "
          f"{pct(tt['correct'],tt['n'])}%   (n={tt['n']}){neu}")

## Next steps

- **Save the adapter to Drive** so it survives the runtime: `from google.colab import
  drive; drive.mount('/content/drive')` then copy `runs/qwen35_0p8b/adapter` across.
- **Headline metric:** **neutral-class recall** on the entailment core (the Uncertain/
  Unknown "does-not-follow" label — the hard part). Because the held-out split covers
  **every** family, read in-distribution neutral recall (FOLIO/ProverQA/LogicNLI)
  together; success = tuned ≥ base on parse rate + accuracy, biggest lift on neutral.
- **Regenerate / rebalance the data:** edit the `CAPS` in `build_corpus.py` and rerun
  `python build_corpus.py` (needs `data/_raw/`). It rewrites `sft_train.jsonl` +
  `eval_items.json`. All completions are one house style; ProverQA uses its
  prover-derived chain, the rest use grounded templates.
- **Higher-fidelity traces (optional upgrade):** replace the grounded templates with a
  frontier verbalizer (translate-only, grounded in each item's derivation) — see
  `../PLAN.md` → "Day 3+" and the parent `gen_data.py`. Keep it translate-only so it
  does not reintroduce hand-wavy `Unknown` rationalization.
- **Model id:** if `Qwen/Qwen3.5-0.8B` is unavailable, set `MODEL_ID` to the exact id
  from the [Qwen HF org](https://huggingface.co/Qwen) (fallback: `Qwen/Qwen3-0.6B`).